### 定义超参数

In [1]:
from transformers import PretrainedConfig

class ModelConfig(PretrainedConfig): 
    def __init__(
        self,
        dim: int=768, # 模型维度
        n_layers: int=12, # transformer的层数
        n_heads: int=16, # 注意力机制的头数
        n_kv_heads: int=8, # 键值头的数量，此处2个查询头共享一个键值头
        vocab_size: int=6144, # 词汇表大小
        hidden_dim: int=None, # 隐藏层维度
        multiple_of: int=64, # 多个of, 用于调整隐藏层维度，使其是该值的倍数
        norm_eps: float=1e-5, # 归一化层的eps
        max_seq_len: int=512, # 最大序列长度
        dropout: float=0.0, # dropout概率
        flash_attn: bool=True, # 是否使用flash attention
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.multiple_of = multiple_of
        self.norm_eps = norm_eps
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.flash_attn = flash_attn
        

/root/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Llama2模型架构

#### 构建RMSNorm

In [2]:
# RMSNorm归一化有助于通过确保权重的规模不会变得过大或过小来稳定学习过程
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    
    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output*self.weight

In [3]:
# 测试
args = ModelConfig()
norm = RMSNorm(args.dim, args.norm_eps)
x = torch.randn(1, 50, args.dim) # (batch_size, seq_len, dim)
output = norm(x)
print(output.shape)

torch.Size([1, 50, 768])


#### 构建Llama2 attenion

In [4]:
# repeat_kv
# 因为使用了GQA机制，需要将键和值的维度扩展到和查询的维度一样才能进行计算
# 此处查询头的数量是键值头的两倍，所以需要将键值头的权重复制一份，以供查询头使用

In [5]:
def repeat_kv(x, n_rep):
    ''' 
    x: (batch_size, seq_len, n_kv_heads, head_dim)
    n_rep: n_heads // n_kv_heads
    '''
    bs, seq_len, n_kv_heads, head_dim = x.shape

    if n_rep == 1: # 如果查询头和键值头数量相同，则不需要复制
        return x
    # 复制键值头的权重，使其数量和查询头一样
    # repeat_interleave() 是 PyTorch 中用于沿指定维度重复张量元素的函数
    return x.repeat_interleave(n_rep, dim=2) # (batch_size, seq_len, n_kv_heads*n_rep=n_heads, head_dim)

In [6]:
# 测试
args = ModelConfig()
x1 = torch.randn(1, 50, args.n_kv_heads, args.dim // args.n_heads) # (batch_size, seq_len, n_kv_heads, head_dim)
output = repeat_kv(x1, args.n_heads // args.n_kv_heads)
print(output.shape)

torch.Size([1, 50, 16, 48])


In [7]:
# 旋转位置嵌入
# 旋转查询（Q）和键（K）向量来注入相对位置信息，使得注意力机制能够自然地捕捉token之间的距离关系。

In [8]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    预计算旋转位置嵌入的复数频率矩阵
    
    参数:
    dim: 每个头的维度 (必须是偶数)
    end: 最大序列长度
    theta: 基频参数
    
    返回:
    freqs_cis: 复数张量，形状为 (end, dim//2)
    """
    # 确保dim是偶数
    assert dim % 2 == 0, f"dim必须是偶数，但得到{dim}"
    
    # 计算每个维度的基础频率
    indices = torch.arange(0, dim, 2)[: (dim // 2)].float()
    freqs = 1.0 / (theta ** (indices / dim))
    
    # 生成位置序列
    t = torch.arange(end, device=freqs.device)
    
    # 计算外积: m * θ_i
    freqs = torch.outer(t, freqs).float()  # 形状: (end, dim//2)
    
    # 转换为复数形式: cosθ + i*sinθ
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    
    return freqs_cis

class RotaryPositionEmbedding:
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        assert dim % 2 == 0, f"dim必须是偶数，但得到{dim}"
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        
        # 预计算复数频率矩阵
        self.freqs_cis = precompute_freqs_cis(dim, max_seq_len, base)
    
    def apply_rotary_emb(self, x: torch.Tensor, start_pos: int = 0):
        """
        应用旋转位置嵌入
        
        参数:
        x: 输入张量，形状为 (batch_size, seq_len, n_heads, head_dim)
        start_pos: 起始位置，用于增量解码
        
        返回:
        旋转后的张量，形状与x相同
        """
        batch_size, seq_len, n_heads, head_dim = x.shape
        
        # 验证维度
        assert head_dim == self.dim, f"head_dim({head_dim})必须等于初始化时的dim({self.dim})"
        assert head_dim % 2 == 0, f"head_dim必须是偶数，但得到{head_dim}"
        
        # 重塑为复数形式: (batch, seq_len, n_heads, head_dim//2, 2) -> 复数
        x_reshaped = x.float().reshape(batch_size, seq_len, n_heads, head_dim // 2, 2)
        x_complex = torch.view_as_complex(x_reshaped)
        
        # 获取对应的旋转因子（确保与x在同一设备与dtype）
        freqs_cis = self.freqs_cis.to(device=x.device, dtype=x_complex.dtype)
        freqs_cis = freqs_cis[start_pos:start_pos + seq_len]
        freqs_cis = freqs_cis.view(1, seq_len, 1, head_dim // 2)
        
        # 应用旋转: 复数乘法
        x_rotated_complex = x_complex * freqs_cis
        
        # 转换回实数形式
        x_rotated = torch.view_as_real(x_rotated_complex)
        x_rotated = x_rotated.reshape(batch_size, seq_len, n_heads, head_dim)
        
        return x_rotated.type_as(x)

In [9]:
# 测试
x = torch.randn(1, 50, 16, 48) # (batch_size, seq_len, n_heads, head_dim)
# 初始化RoPE
rope = RotaryPositionEmbedding(dim=args.dim // args.n_heads, max_seq_len=args.max_seq_len)
    
# 应用RoPE
x_rotated = rope.apply_rotary_emb(x)
print(f"输出形状: {x_rotated.shape}")


输出形状: torch.Size([1, 50, 16, 48])


In [10]:
# 组装 Attention

In [11]:
class Attention(nn.Module):
    def __init__(self, args:ModelConfig):
        super().__init__()
        # 根据是否指定n_kv_heads,确定用于键值头的数量
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        # 确保总头数可以被键值头数整除
        assert args.n_heads % self.n_kv_heads == 0, f"n_heads({args.n_heads})必须能被n_kv_heads({self.n_kv_heads})整除"

        # 模型并行处理大小，默认为1
        model_parallel_size = getattr(args, "model_parallel_size", 1)
        # 本地计算头数，等于总头数除以模型并行处理大小
        self.n_local_heads = args.n_heads // model_parallel_size
        # 本地键值头数，等于键值头数除以模型并行处理大小
        self.n_local_kv_heads = self.n_kv_heads // model_parallel_size
        # 重复次数，用于扩展键和值的尺寸
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        # 每个头的维度，等于模型维度除以总头数
        self.head_dim = args.dim // args.n_heads

        # 定义权重矩阵
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)

        # 定义dropout
        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)
        # 保存dropout概率
        self.dropout = args.dropout

        # 检查是否使用flash attention
        self.flash_attn = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash_attn:
            print("警告: 当前PyTorch版本不支持flash attention，将使用常规注意力实现，性能可能较差。请升级到PyTorch 2.0或更高版本以获得最佳性能。")
            # 创建一个上三角矩阵，用于掩盖未来的时间步
            mask = torch.full((1,1,args.max_seq_len,args.max_seq_len), float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            # 注册为缓冲区，以便在模型保存和加载时自动处理
            self.register_buffer("mask", mask)

    def forward(self, x):
        '''
        输入x的形状为 (batch_size, seq_len, dim)
        输出的形状为 (batch_size, seq_len, dim)
        '''
        # 获取批次大小和序列长度
        batch_size, seq_len, _ = x.shape

        # 计算查询、键和值
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        # 重塑为多头格式
        xq = xq.view(batch_size, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(batch_size, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(batch_size, seq_len, self.n_local_kv_heads, self.head_dim)

        # 应用旋转位置嵌入
        rope = RotaryPositionEmbedding(dim=self.head_dim, max_seq_len=seq_len)
        xq = rope.apply_rotary_emb(xq)
        xk = rope.apply_rotary_emb(xk)

        # 扩展键和值的尺寸以匹配查询的尺寸
        xk = repeat_kv(xk, self.n_rep)
        xv = repeat_kv(xv, self.n_rep)

        # 将头作为批次维度处理
        xq = xq.transpose(1, 2) # (batch_size, n_local_heads, seq_len, head_dim)
        xk = xk.transpose(1, 2) # (batch_size, n_local_heads, seq_len, head_dim)
        xv = xv.transpose(1, 2) # (batch_size, n_local_heads, seq_len, head_dim)

        # 根据是否支持flash attention选择计算方法
        if self.flash_attn:
            # 使用flash attention计算注意力输出
            attn_output = torch.nn.functional.scaled_dot_product_attention(
                xq, xk, xv, attn_mask=None if self.dropout == 0.0 else self.mask[:,:,:seq_len,:seq_len], dropout_p=self.dropout, is_causal=True
            )
        else:
            # 使用手动实现注意力计算
            attn_scores = torch.matmul(xq, xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            assert hasattr(self, "mask")
            attn_scores = attn_scores + self.mask[:,:,:seq_len,:seq_len]
            attn_probs = torch.softmax(attn_scores, dim=-1)
            attn_probs = self.attn_dropout(attn_probs)
            attn_output = torch.matmul(attn_probs, xv)

        # 转置回原始格式
        attn_output = attn_output.transpose(1, 2).contiguous() # (batch_size, seq_len, n_local_heads, head_dim)
        attn_output = attn_output.view(batch_size, seq_len, self.n_local_heads * self.head_dim) # (batch_size, seq_len, dim)

        # 通过输出线性层
        output = self.wo(attn_output)
        output = self.resid_dropout(output)
        return output

In [12]:
# 测试
args = ModelConfig()
attention = Attention(args)
x = torch.randn(1, 50, args.dim) # (batch_size, seq_len, dim)
output = attention(x)
print(output.shape)

torch.Size([1, 50, 768])


#### 构建MLP模块

In [13]:
class MLP(nn.Module):
    '''
    多层感知机（MLP）模块，包含两个线性层和一个激活函数
    输入维度为模型维度，输出维度也为模型维度，中间隐藏层的维度可以通过hidden_dim参数指定
    如果hidden_dim未指定，则默认为模型维度的4倍，并且调整为multiple_of的倍数
    该模块包含dropout以防止过拟合,激活函数使用GELU，适用于Transformer架构
    输入和输出的形状均为 (batch_size, seq_len, dim)
    '''
    def __init__(self, args:ModelConfig):
        super().__init__()
        # 如果未指定hidden_dim，则默认为模型维度的4倍，并调整为multiple_of的倍数
        hidden_dim = args.hidden_dim or (4 * args.dim)
        if hidden_dim % args.multiple_of != 0:
            hidden_dim = ((hidden_dim + args.multiple_of - 1) // args.multiple_of) * args.multiple_of
        
        self.fc1 = nn.Linear(args.dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, args.dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [14]:
# 测试
mlp = MLP(args)
x = torch.randn(1, 50, args.dim) # (batch_size, seq_len, dim)
output = mlp(x)
print(output.shape)

torch.Size([1, 50, 768])


#### 构建Decoder Layer

In [15]:
class DecoderLayer(nn.Module):
    '''
    解码器层，包含一个自注意力模块和一个前馈网络（MLP）
    输入和输出的形状均为 (batch_size, seq_len, dim)
    '''
    def __init__(self, layer_id, args:ModelConfig):
        super().__init__()
        # 定义多头注意力的头数
        self.n_heads = args.n_heads
        # 定义输入维度
        self.dim = args.dim
        # 定义每个头的维度
        self.head_dim = self.dim // self.n_heads
        # 定义自注意力模块和前馈网络
        self.attention = Attention(args)
        self.feed_forwad = MLP(args)
        # 定义层的id
        self.layer_id = layer_id
        # 定义注意力计算的归一化
        self.attn_norm = RMSNorm(args.dim, args.norm_eps)
        # 定义前馈网络的归一化
        self.ffn_norm = RMSNorm(args.dim, args.norm_eps)

    def forward(self, x):
        # 输入x经过注意力归一化层，进行注意力计算，结果与x相加形成残差连接
        attn_output = self.attention(self.attn_norm(x))
        h = x + attn_output
        # x经过前馈网络归一化层，进行前馈计算，结果与x相加形成残差连接
        ffn_output = self.feed_forwad(self.ffn_norm(h))
        out = h + ffn_output
        return out


In [16]:
# 测试
args = ModelConfig()
decoderlayer = DecoderLayer(0, args)
x = torch.randn(1, 50, args.dim) # (batch_size, seq_len, dim)
output = decoderlayer(x)
print(output.shape)

torch.Size([1, 50, 768])


#### 构建Llama2模型

In [17]:
from transformers import PreTrainedModel
import math
from transformers.modeling_outputs import CausalLMOutputWithPast

class Llama2(PreTrainedModel):
    '''
    Llama2模型类，继承自PreTrainedModel
    包含一个解码器层列表，层数由ModelConfig中的n_layers指定
    输入和输出的形状均为 (batch_size, seq_len, dim)
    '''
    def __init__(self, args:ModelConfig):
        super().__init__(args)
        # 初始化模型参数
        self.args = args
        # 词汇表大小
        self.vocab_size = args.vocab_size
        # 层数
        self.n_layers = args.n_layers
        
        # 词嵌入层
        self.token_embeddings = nn.Embedding(self.vocab_size, args.dim)
        # Dropout层
        self.dropout = nn.Dropout(args.dropout)
        # 解码器层列表
        self.layers = nn.ModuleList()
        for layer_id in range(self.n_layers):
            self.layers.append(DecoderLayer(layer_id, args))
        # 归一化层
        self.norm = RMSNorm(args.dim, args.norm_eps)
        # 输出线性层，将模型维度映射回词汇表大小
        self.output_projection = nn.Linear(args.dim, self.vocab_size, bias=False)

        # 将词嵌入层的权重与输出线性层的权重共享
        self.output_projection.weight = self.token_embeddings.weight

        # 预计算相对位置嵌入的复数频率矩阵
        self.rope = RotaryPositionEmbedding(dim=args.dim // args.n_heads, max_seq_len=args.max_seq_len)

        # 初始化所有权重
        self.apply(self._init_weights)
        # 对残差投影权重进行特殊初始化
        for pn, p in self.named_parameters():
            if pn.endswith("wo.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.n_layers))

        # 初始化最后一次前向传播的损失属性
        self.last_forward_loss = None
        self.OUT = CausalLMOutputWithPast() # 输出容器
        self._no_split_modules = [name for name, _ in self.named_modules()] # 模块名称列表，用于模型并行处理时避免拆分特定模块

    def _init_weights(self, module):
        # 初始化权重函数
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens, targets=None, **kwargs):
        '''
        前向传播函数
        tokens: 输入的token ids，形状为 (batch_size, seq_len)
        targets: 目标token ids，形状为 (batch_size, seq_len)，用于计算损失
        返回值: 包含logits和loss的字典
        '''
        if 'input_ids' in kwargs:
            tokens = kwargs['input_ids']
        if 'labels' in kwargs:
            targets = kwargs['labels']

        # 获取批次大小和序列长度
        batch_size, seq_len = tokens.shape
        # 计算token嵌入
        x = self.token_embeddings(tokens) # (batch_size, seq_len, dim)
        x = self.dropout(x)
        # 相对位置嵌入已经在Attention模块中应用，因此这里不需要再次应用
        # 依次通过每个解码器层
        for layer in self.layers:
            x = layer(x) # (batch_size, seq_len, dim)
        # 通过归一化层
        x = self.norm(x) # (batch_size, seq_len, dim)

        if targets is not None:
            # 计算logits
            logits = self.output_projection(x) # (batch_size, seq_len, vocab_size)
            # 计算损失
            loss_fct = nn.CrossEntropyLoss(ignore_index=0, reduction='none')
            self.last_loss = loss_fct(logits.view(-1, logits.size(-1)), targets.view(-1))
            
        else:
            # 推理时的小优化：只对最后一个位置的输出进行前向传播
            logits = self.output_projection(x[:, [-1], :]) 
            self.last_loss = None

        # 返回logits和loss
        self.OUT.__setitem__('logits', logits)
        self.OUT.__setitem__('last_loss', self.last_loss)
        return self.OUT

    # 推理
    @torch.inference_mode()
    def generate(self, idx, stop_id=None, max_new_tokens=256, temperature=1.0, top_k=None):
        """
        给定输入序列 idx（形状为 (bz,seq_len) 的长整型张量），通过多次生成新 token 来完成序列。
        在 model.eval() 模式下运行。效率较低的采样版本，没有使用键k/v cache。
        """
        index = idx.shape[1]
        for _ in range(max_new_tokens):
            # 如果序列上下文过长，截断它到最大长度
            idx_cond = idx if idx.size(1) <= self.args.max_seq_len else idx[:, -self.args.max_seq_len:]
            
            # 前向传播获取序列中最后一个位置的 logits
            logits = self(idx_cond).logits
            logits = logits[:, -1, :] # 只保留最后一个时间步的输出
            
            if temperature == 0.0:
                # 选择最有可能的索引
                _, idx_next = torch.topk(logits, k=1, dim=-1)
            else:
                # 缩放 logits 并应用 softmax
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
            

            if idx_next == stop_id:
                break

            # 将采样的索引添加到序列中并继续
            idx = torch.cat((idx, idx_next), dim=1)

        return idx[:, index:] # 只返回生成的token

In [18]:
# 测试
args = ModelConfig()
model = Llama2(args)
tokens = torch.randint(0, args.vocab_size, (1, 50)) # (batch_size, seq_len)
output = model(tokens)
# 计算模型的全部参数
num_params = sum(p.numel() for p in model.parameters())
print(f'模型总参数量: {num_params}')
print(output.logits.shape) # (batch_size, 1, vocab_size),因为设计了模型只返回最后一个位置的输出  

模型总参数量: 82640640
torch.Size([1, 1, 6144])


### 训练Tokenizer

In [19]:
# BPE算法训练Subword Tokenizer
# 使用Hugging Face的tokenizers库来训练一个基于BPE算法的子词分词器

In [19]:
import random
import json
import os
from transformers import PreTrainedTokenizerFast, AutoTokenizer
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer
from tokenizers.normalizers import NFKC
from typing import Generator


In [42]:
# 使用出门问问序列猴子开源数据集训练tokenizer
# 下载预训练数据集
os.system("modelscope download --dataset ddzhu123/seq-monkey mobvoi_seq_monkey_general_open_corpus.jsonl.tar.bz2 --local_dir /fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data")
# 解压预训练数据集
os.system("tar -xvf /fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/mobvoi_seq_monkey_general_open_corpus.jsonl.tar.bz2")



 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'




Successfully Downloaded from dataset ddzhu123/seq-monkey.

mobvoi_seq_monkey_general_open_corpus.jsonl


0

In [20]:
# 加载训练数据
def read_texts_from_jsonl(file_path):
    """读取JSONL文件并安全提取文本数据"""
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                data = json.loads(line)
                if 'text' not in data:
                    raise KeyError(f"Missing 'text' field in line {line_num}")
                yield data['text']
            except json.JSONDecodeError:
                print(f"Error decoding JSON in line {line_num}")
                continue
            except KeyError as e:
                print(e)
                continue

In [21]:
# 创建配置文件
def create_tokenizer_config(save_dir):
    """创建完整的tokenizer配置文件"""
    config = {
        "add_bos_token": False,
        "add_eos_token": False,
        "add_prefix_space": False,
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "pad_token": "<|im_end|>",
        "unk_token": "<unk>",
        "model_max_length": 1000000000000000019884624838656,
        "clean_up_tokenization_spaces": False,
        "tokenizer_class": "PreTrainedTokenizerFast",
        "chat_template": (
            "{% for message in messages %}"
            "{% if message['role'] == 'system' %}"
            "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'user' %}"
            "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'assistant' %}"
            "<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
            "{% endif %}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|im_start|>assistant\n' }}"
            "{% endif %}"
        )
    }

    # 保存主配置文件
    with open(os.path.join(save_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)

    # 创建special_tokens_map.json
    special_tokens_map = {
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "unk_token": "<unk>",
        "pad_token": "<|im_end|>",
        "additional_special_tokens": ["<s>", "</s>"]
    }
    with open(os.path.join(save_dir, "special_tokens_map.json"), "w", encoding="utf-8") as f:
        json.dump(special_tokens_map, f, ensure_ascii=False, indent=4)

In [22]:
# 训练tokenizer
def train_tokenizer(data_path, save_dir, vocab_size=8192) -> None:
    """训练并保存自定义tokenizer"""
    os.makedirs(save_dir, exist_ok=True)
    
    # 初始化tokenizer
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = NFKC()  # 添加文本规范化
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    # 配置特殊token
    special_tokens = [
        "<unk>", 
        "<s>", 
        "</s>", 
        "<|im_start|>", 
        "<|im_end|>"
    ]

    # 配置训练器
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        min_frequency=2,  # 提高低频词过滤
        show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
    )

    # 训练tokenizer
    print(f"Training tokenizer with data from {data_path}")
    texts = read_texts_from_jsonl(data_path)
    tokenizer.train_from_iterator(texts, trainer=trainer, length=os.path.getsize(data_path))

    # 验证特殊token映射
    try:
        assert tokenizer.token_to_id("<unk>") == 0
        assert tokenizer.token_to_id("<s>") == 1
        assert tokenizer.token_to_id("</s>") == 2
        assert tokenizer.token_to_id("<|im_start|>") == 3
        assert tokenizer.token_to_id("<|im_end|>") == 4
    except AssertionError as e:
        print("Special tokens mapping error:", e)
        raise

    # 保存tokenizer文件
    tokenizer.save(os.path.join(save_dir, "tokenizer.json"))
    
    # 创建配置文件
    create_tokenizer_config(save_dir)
    print(f"Tokenizer saved to {save_dir}")

In [46]:
# 训练tokenizer
save_dir = "/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/tokenizer"
data_path = "/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/mobvoi_seq_monkey_general_open_corpus.jsonl"
train_tokenizer(data_path, save_dir, vocab_size=args.vocab_size)

Training tokenizer with data from /fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/mobvoi_seq_monkey_general_open_corpus.jsonl



: 

In [ ]:
# 修改为在train_tokenizer.py中训练

### 处理预训练数据集

In [23]:
from torch.utils.data import Dataset
import numpy as np


class PretrainDataset(Dataset):
    '''
    预训练数据集类，继承自torch.utils.data.Dataset
    用于加载和处理预训练数据，返回输入和目标token ids
    '''
    def __init__(self, data_path, tokenizer, max_seq_len=512):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token is not None else 0
        # 预计算每行的起始字节偏移量
        self._offsets = [] #记录了文件中每一行的起始字节位置
        with open(data_path, 'rb') as f:
            self._offsets.append(0)
            while f.readline():
                self._offsets.append(f.tell())
        self._total_lines = len(self._offsets) - 1  # 最后一个 tell() 是 EOF

    def __len__(self):
        # 计算数据集的长度，即文件中的行数
        return self._total_lines
    
    def __getitem__(self, idx):
        with open(self.data_path, 'rb') as f:
            # 跳到第idx个样本的位置
            f.seek(self._offsets[idx])
            line = f.readline().decode('utf-8')
        sample = json.loads(line)
        text = f'{self.tokenizer.bos_token}{sample["text"]}' # 添加BOS token
        # 使用tokenizer编码文本，返回input_ids
        input_id = self.tokenizer(text).data['input_ids'][:self.max_seq_len]
        text_len = len(input_id)

        # 没满最大长度的剩余部分
        padding_len = self.max_seq_len - text_len
        input_id = input_id + [self.padding] * padding_len
        # 0 表示不计算损失
        loss_mask = [1] * text_len + [0] * padding_len

        input_id =np.array(input_id, dtype=np.int64)
        X = np.array(input_id[:-1], dtype=np.int64) # 输入序列，去掉最后一个token
        Y = np.array(input_id[1:], dtype=np.int64) # 目标序列，去掉第一个token
        loss_mask = np.array(loss_mask[1:], dtype=np.int64) # 损失掩码，去掉第一个token
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

### 训练模型

In [24]:
from transformers import AutoTokenizer
import json
import os


def get_lr(it, all):
    """
    计算当前迭代的学习率，使用余弦退火调度策略
    
    学习率调度策略：
    1. Warmup阶段：学习率从0线性增长到目标学习率
    2. 余弦退火阶段：学习率按余弦函数衰减到最小学习率
    3. 超出训练步数后：保持最小学习率
    
    Args:
        it (int): 当前迭代步数
        all (int): 总迭代步数
        
    Returns:
        float: 当前步数对应的学习率
    """
    warmup_iters = args.warmup_iters  # 预热迭代次数
    lr_decay_iters = all  # 学习率衰减的总迭代次数
    min_lr = args.learning_rate / 10  # 最小学习率，为初始学习率的1/10

    # Warmup阶段：线性增长
    if it < warmup_iters:
        return args.learning_rate * it / warmup_iters
    
    # 超出训练步数：保持最小学习率
    if it > lr_decay_iters:
        return min_lr
    
    # 余弦退火阶段
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # 余弦系数
    return min_lr + coeff * (args.learning_rate - min_lr)

def train_epoch(epoch):
    """
    训练一个epoch的函数
    
    实现了完整的训练循环，包括：
    1. 数据加载和设备转移
    2. 动态学习率调整
    3. 前向传播和损失计算
    4. 梯度累积和反向传播
    5. 梯度裁剪和优化器更新
    6. 日志记录和模型保存
    
    Args:
        epoch (int): 当前epoch编号
    """
    start_time = time.time()  # 记录开始时间
    
    # 遍历数据加载器中的每个batch
    for step, (X, Y, loss_mask) in enumerate(train_loader):
        # 将数据转移到指定设备（GPU/CPU）
        X = X.to(args.device)  # 输入序列
        Y = Y.to(args.device)  # 目标序列
        loss_mask = loss_mask.to(args.device)  # 损失掩码，用于忽略padding token

        # 计算当前步骤的学习率
        lr = get_lr(epoch * iters_per_epoch + step, args.epochs * iters_per_epoch)
        # 更新优化器中所有参数组的学习率
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # 使用混合精度训练上下文
        with ctx:
            # 前向传播
            out = model(X, Y)
            # 计算损失并除以累积步数（用于梯度累积）
            loss = out.last_loss / args.accumulation_steps
            # 将loss_mask展平为一维
            loss_mask = loss_mask.view(-1)
            # 应用掩码计算有效损失（忽略padding位置）
            loss = torch.sum(loss * loss_mask) / loss_mask.sum()

        # 使用scaler进行混合精度的反向传播
        scaler.scale(loss).backward()

        # 每accumulation_steps步执行一次优化器更新
        if (step + 1) % args.accumulation_steps == 0:
            # 取消梯度缩放，准备梯度裁剪
            scaler.unscale_(optimizer)
            # 梯度裁剪，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            # 执行优化器步骤
            scaler.step(optimizer)
            # 更新scaler的缩放因子
            scaler.update()

            # 清零梯度，set_to_none=True可以节省内存
            optimizer.zero_grad(set_to_none=True)

        # 每log_interval步记录一次日志
        if step % args.log_interval == 0:
            spend_time = time.time() - start_time
            # 打印训练进度信息
            Logger(
                'Epoch:[{}/{}]({}/{}) loss:{:.3f} lr:{:.7f} epoch_Time:{}min;'.format(
                    epoch + 1,
                    args.epochs,
                    step,
                    iters_per_epoch,
                    loss.item() * args.accumulation_steps,  # 恢复真实的loss值
                    optimizer.param_groups[-1]['lr'],
                    spend_time / (step + 1) * iters_per_epoch // 60 - spend_time // 60))
            
            # 如果启用SwanLab，记录训练指标
            if args.use_swanlab:
                swanlab.log({
                    "loss": loss.item() * args.accumulation_steps,
                    "lr": optimizer.param_groups[-1]['lr']
                })

        # 每save_interval步保存一次模型
        if (step + 1) % args.save_interval == 0:
            model.eval()  # 切换到评估模式
            # 构建检查点文件名
            ckp = f'{args.save_dir}/pretrain_{lm_config.dim}_{lm_config.n_layers}_{lm_config.vocab_size}.pth'

            # 处理多卡保存：如果是DataParallel模型，需要访问.module属性
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()  # 切换回训练模式
        
        # 每20000步保存一个带步数标记的检查点
        if (step + 1) % 20000 == 0:
            model.eval()
            # 构建带步数的检查点文件名
            ckp = f'{args.save_dir}/pretrain_{lm_config.dim}_{lm_config.n_layers}_{lm_config.vocab_size}_step{step+1}.pth'

            # 保存模型状态字典
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()


def init_model():
    """
    初始化模型和分词器
    
    功能包括：
    1. 加载预训练的分词器
    2. 创建Transformer模型
    3. 设置多GPU并行训练（如果可用）
    4. 将模型移动到指定设备
    5. 统计并打印模型参数量
    
    Returns:
        tuple: (model, tokenizer) 初始化后的模型和分词器
    """
    def count_parameters(model):
        """
        统计模型中可训练参数的数量
        
        Args:
            model: PyTorch模型
            
        Returns:
            int: 可训练参数总数
        """
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 从本地路径加载预训练的分词器
    tokenizer = AutoTokenizer.from_pretrained('/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/tokenizer')

    # 根据配置创建Transformer模型
    model = Llama2(lm_config)
    
    # 多卡初始化：检查可用GPU数量并设置DataParallel
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        Logger(f"Using {num_gpus} GPUs with DataParallel!")
        # 使用DataParallel包装模型以支持多GPU训练
        model = torch.nn.DataParallel(model)
    
    # 将模型移动到指定设备（GPU或CPU）
    model = model.to(args.device)
    
    # 计算并打印模型参数量（以百万为单位）
    Logger(f'LLM总参数量：{count_parameters(model) / 1e6:.3f} 百万')
    return model, tokenizer

In [35]:
# 执行训练（确保上面的单元已运行完）
import time
from contextlib import nullcontext
from torch.utils.data import DataLoader
import torch.nn.functional as F

# 训练相关超参数（如果之前没定义就补上）
if not hasattr(args, "device"):
    args.device = "cuda" if torch.cuda.is_available() else "cpu"
if not hasattr(args, "train_data_path"):
    args.train_data_path = "/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/mobvoi_seq_monkey_general_open_corpus.jsonl"
if not hasattr(args, "batch_size"):
    args.batch_size = 8
if not hasattr(args, "num_workers"):
    args.num_workers = 2
if not hasattr(args, "epochs"):
    args.epochs = 1
if not hasattr(args, "learning_rate"):
    args.learning_rate = 3e-4
if not hasattr(args, "warmup_iters"):
    args.warmup_iters = 100
if not hasattr(args, "accumulation_steps"):
    args.accumulation_steps = 4
if not hasattr(args, "max_grad_norm"):
    args.max_grad_norm = 1.0
if not hasattr(args, "grad_clip"):
    args.grad_clip = args.max_grad_norm
if not hasattr(args, "log_interval"):
    args.log_interval = 50
if not hasattr(args, "save_interval"):
    args.save_interval = 2000
if not hasattr(args, "use_swanlab"):
    args.use_swanlab = False
if not hasattr(args, "save_dir"):
    args.save_dir = "/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/output"

# 训练日志函数（如果之前没定义就补上）
if "Logger" not in globals():
    def Logger(msg):
        print(msg)

# 确保训练用的配置存在
if "lm_config" not in globals():
    lm_config = args

# 初始化模型与分词器
model, tokenizer = init_model()

# 数据加载器
train_dataset = PretrainDataset(args.train_data_path, tokenizer, max_seq_len=args.max_seq_len)
train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.num_workers,
    pin_memory=True,
)

iters_per_epoch = len(train_loader)

# 优化器与混合精度
optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
use_amp = args.device.startswith("cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()

# 开始训练
for epoch in range(args.epochs):
    train_epoch(epoch)

Logger("训练结束")

Using 4 GPUs with DataParallel!
LLM总参数量：82.641 百万


/tmp/ipykernel_28233/408570861.py:64: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Epoch:[1/1](0/1625000) loss:8.815 lr:0.0000000 epoch_Time:35485.0min;
Epoch:[1/1](50/1625000) loss:7.567 lr:0.0001500 epoch_Time:9216.0min;
Epoch:[1/1](100/1625000) loss:7.126 lr:0.0003000 epoch_Time:6010.0min;
Epoch:[1/1](150/1625000) loss:6.737 lr:0.0003000 epoch_Time:4886.0min;
Epoch:[1/1](200/1625000) loss:4.911 lr:0.0003000 epoch_Time:4321.0min;
Epoch:[1/1](250/1625000) loss:6.207 lr:0.0003000 epoch_Time:4008.0min;
Epoch:[1/1](300/1625000) loss:7.059 lr:0.0003000 epoch_Time:3773.0min;
Epoch:[1/1](350/1625000) loss:7.510 lr:0.0003000 epoch_Time:3623.0min;
Epoch:[1/1](400/1625000) loss:6.525 lr:0.0003000 epoch_Time:3495.0min;
Epoch:[1/1](450/1625000) loss:7.587 lr:0.0003000 epoch_Time:3414.0min;
Epoch:[1/1](500/1625000) loss:4.913 lr:0.0003000 epoch_Time:3347.0min;
Epoch:[1/1](550/1625000) loss:5.909 lr:0.0003000 epoch_Time:3278.0min;
Epoch:[1/1](600/1625000) loss:4.431 lr:0.0003000 epoch_Time:3234.0min;
Epoch:[1/1](650/1625000) loss:7.292 lr:0.0003000 epoch_Time:3192.0min;
Epoch:[1

### 处理微调数据集

In [25]:
import json
from tqdm import tqdm

def convert_message(data):
    """
    将原始数据转换为标准格式
    """
    message = [
        {"role": "system", "content": "你是一个AI助手"},
    ]
    for item in data:
        if item['from'] == 'human':
            message.append({'role': 'user', 'content': item['value']})
        elif item['from'] == 'assistant':
            message.append({'role': 'assistant', 'content': item['value']})
    return message

with open('/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/BelleGroup_sft.jsonl', 'a', encoding='utf-8') as sft:
    with open('/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/train_3.5M_CN.json', 'r', encoding='utf-8') as f:
        data = f.readlines()
        for item in tqdm(data, desc="Processing", unit="lines"):
            item = json.loads(item)
            message = convert_message(item['conversations'])
            sft.write(json.dumps(message, ensure_ascii=False) + '\n')

Processing: 100%|██████████| 3606402/3606402 [01:08<00:00, 52414.57lines/s]


In [30]:
class SFTDataset(Dataset):
    '''
    SFT数据集类，继承自torch.utils.data.Dataset
    用于加载和处理SFT数据
    输入是上一轮的对话内容，输出是当前轮的对话内容
    '''
    def __init__(self, data_path, tokenizer, max_seq_len=512):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token is not None else 0
        # 预计算每行的起始字节偏移量
        self._offsets = []
        with open(data_path, 'rb') as f:
            self._offsets.append(0)
            while f.readline():
                self._offsets.append(f.tell())
        self._total_lines = len(self._offsets) - 1  # 最后一个 tell() 是 EOF

    def __len__(self):
        # 计算数据集的长度，即文件中的行数
        return self._total_lines
    
    def generate_loss_mask(self, input_id):
        # 生成损失掩码，1表示计算损失，0表示不计算损失
        mask = [0] * len(input_id)
        a_sequence = self.tokenizer("<|im_start|>assistant\n")['input_ids']
        a_len = len(a_sequence)
        n = len(input_id)
        i = 0

        while i <= n - a_len:
            match = True
            for k in range(a_len):
                if input_id[i + k] != a_sequence[k]:
                    match = False
                    break
            if match:
                # 从子序列结束的位置开始查找第一个 4 (eos_token_id)
                j = None
                for idx in range(i + a_len, n):
                    if input_id[idx] == self.tokenizer.eos_token_id:
                        j = idx
                        break
                if j is not None:
                    start = i + a_len
                    end = j
                    # 标记区间为1（包括start和end）
                    if start <= end:
                        for pos in range(start, end + 1):
                            if pos < len(mask):
                                mask[pos] = 1
                
                i += a_len  # 跳过当前子序列
            else:
                i += 1  # 继续检查下一个位置
        return mask
    
    def __getitem__(self, idx):
        with open(self.data_path, 'rb') as f:
            f.seek(self._offsets[idx])
            line = f.readline().decode('utf-8')
        sample = json.loads(line)
        text = self.tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)
        input_id = self.tokenizer(text).data['input_ids'][:self.max_seq_len]
        text_len = len(input_id)
        # 没满最大长度的剩余部分
        padding_len = self.max_seq_len - text_len
        input_id = input_id + [self.padding] * padding_len
        # 0表示不计算损失
        loss_mask = self.generate_loss_mask(input_id)

        input_id = np.array(input_id)
        X = np.array(input_id[:-1]).astype(np.int64)
        Y = np.array(input_id[1:]).astype(np.int64)
        loss_mask = np.array(loss_mask[1:]).astype(np.int64)
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

### 微调模型

In [31]:
import os
import platform
import argparse
import time
import warnings
import math
import pandas as pd
import torch
from torch import optim
from torch.utils.data import DataLoader
from contextlib import nullcontext

from transformers import AutoTokenizer
import swanlab


# 忽略警告
warnings.filterwarnings('ignore')


def Logger(content):
    """日志记录器"""
    print(content)

def get_lr(it, all):
    """获取学习率"""
    # 1) linear warmup for warmup_iters steps
    # 1) 预热迭代的线性预热
    warmup_iters = args.warmup_iters
    lr_decay_iters = all
    min_lr = args.learning_rate / 10

    if it < warmup_iters:
        return args.learning_rate * it / warmup_iters
    
    # 2) if it > lr_decay_iters, return min learning rate
    # 2) 如果迭代次数超过学习率衰减迭代次数，则返回最小学习率
    if it > lr_decay_iters:
        return min_lr
    
    # 3) in between, use cosine decay down to min learning rate
    # 3) 在两者之间，使用余弦衰减至最小学习率
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (args.learning_rate - min_lr)

def train_epoch(epoch):
    """训练一个epoch"""
    start_time = time.time()
    for step, (X, Y, loss_mask) in enumerate(train_loader):
        X = X.to(args.device)
        Y = Y.to(args.device)
        loss_mask = loss_mask.to(args.device)

        # 获取学习率并更新优化器
        lr = get_lr(epoch * iters_per_epoch + step, args.epochs * iters_per_epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # 前向传播
        with ctx:
            out = model(X, Y)
            loss = out.last_loss / args.accumulation_steps
            loss_mask = loss_mask.view(-1)
            loss = torch.sum(loss * loss_mask) / loss_mask.sum()

        # 反向传播
        scaler.scale(loss).backward()

        # 更新权重
        if (step + 1) % args.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        # 打印日志
        if step % args.log_interval == 0:
            spend_time = time.time() - start_time
            Logger(
                'Epoch:[{}/{}]({}/{}) loss:{:.3f} lr:{:.7f} epoch_Time:{}min:'.format(
                    epoch + 1,
                    args.epochs,
                    step,
                    iters_per_epoch,
                    loss.item() * args.accumulation_steps,
                    optimizer.param_groups[-1]['lr'],
                    spend_time / (step + 1) * iters_per_epoch // 60 - spend_time // 60))
            if args.use_swanlab:
                swanlab.log({
                    "loss": loss.item() * args.accumulation_steps,
                    "lr": optimizer.param_groups[-1]['lr']
                })

        # 保存模型
        if (step + 1) % args.save_interval == 0:
            model.eval()
            ckp = f'{args.save_dir}/sft_dim{lm_config.dim}_layers{lm_config.n_layers}_vocab_size{lm_config.vocab_size}.pth'

            # 处理多卡保存
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()
        
        # 定期保存模型
        if (step + 1) % 20000 == 0:
            model.eval()
            ckp = f'{args.save_dir}/sft_dim{lm_config.dim}_layers{lm_config.n_layers}_vocab_size{lm_config.vocab_size}_step{step+1}.pth'

            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()


def init_model():
    """初始化模型"""
    def count_parameters(model):
        """计算模型参数量"""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained('/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/tokenizer')

    # 初始化模型
    model = Llama2(lm_config)

    # 加载预训练权重
    ckp = '/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/output/pretrain_768_12_6144.pth'
    state_dict = torch.load(ckp, map_location=args.device)
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict, strict=False)
    
    # 多卡初始化
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        Logger(f"Using {num_gpus} GPUs with DataParallel!")
        model = torch.nn.DataParallel(model)
    
    model = model.to(args.device)
    Logger(f'LLM总参数量：{count_parameters(model) / 1e6:.3f} 百万')
    return model, tokenizer

In [35]:
import time
from contextlib import nullcontext
from torch.utils.data import DataLoader
import torch.nn.functional as F

# 训练相关超参数（如果之前没定义就补上）
if not hasattr(args, "device"):
    args.device = "cuda" if torch.cuda.is_available() else "cpu"
if not hasattr(args, "train_data_path"):
    args.train_data_path = "/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/data/BelleGroup_sft.jsonl"
if not hasattr(args, "batch_size"):
    args.batch_size = 32
if not hasattr(args, "num_workers"):
    args.num_workers = 8
if not hasattr(args, "epochs"):
    args.epochs = 1
if not hasattr(args, "learning_rate"):
    args.learning_rate = 2e-4
if not hasattr(args, "warmup_iters"):
    args.warmup_iters = 100
if not hasattr(args, "accumulation_steps"):
    args.accumulation_steps = 8
if not hasattr(args, "max_grad_norm"):
    args.max_grad_norm = 1.0
if not hasattr(args, "grad_clip"):
    args.grad_clip = args.max_grad_norm
if not hasattr(args, "log_interval"):
    args.log_interval = 100
if not hasattr(args, "save_interval"):
    args.save_interval = 2000
if not hasattr(args, "use_swanlab"):
    args.use_swanlab = False
if not hasattr(args, "save_dir"):
    args.save_dir = "/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/sft_output"
os.makedirs(args.save_dir, exist_ok=True)

# 训练日志函数（如果之前没定义就补上）
if "Logger" not in globals():
    def Logger(msg):
        print(msg)

# 确保训练用的配置存在
if "lm_config" not in globals():
    lm_config = args

# 初始化模型与分词器
model, tokenizer = init_model()

# 数据加载器
train_dataset = SFTDataset(args.train_data_path, tokenizer, max_seq_len=args.max_seq_len)
train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.num_workers,
    pin_memory=True,
)

iters_per_epoch = len(train_loader)

# 优化器与混合精度
optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
use_amp = args.device.startswith("cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()

# 开始训练
for epoch in range(args.epochs):
    train_epoch(epoch)

Logger("训练结束")

Using 4 GPUs with DataParallel!
LLM总参数量：82.641 百万
Epoch:[1/1](0/112701) loss:2.515 lr:0.0000000 epoch_Time:2880.0min:
Epoch:[1/1](100/112701) loss:2.145 lr:0.0002000 epoch_Time:242.0min:
Epoch:[1/1](200/112701) loss:1.998 lr:0.0002000 epoch_Time:221.0min:
Epoch:[1/1](300/112701) loss:1.845 lr:0.0002000 epoch_Time:217.0min:
Epoch:[1/1](400/112701) loss:2.204 lr:0.0002000 epoch_Time:214.0min:
Epoch:[1/1](500/112701) loss:1.718 lr:0.0002000 epoch_Time:213.0min:
Epoch:[1/1](600/112701) loss:1.589 lr:0.0002000 epoch_Time:211.0min:
Epoch:[1/1](700/112701) loss:2.002 lr:0.0002000 epoch_Time:210.0min:
Epoch:[1/1](800/112701) loss:1.703 lr:0.0002000 epoch_Time:209.0min:
Epoch:[1/1](900/112701) loss:1.777 lr:0.0002000 epoch_Time:208.0min:
Epoch:[1/1](1000/112701) loss:1.996 lr:0.0002000 epoch_Time:208.0min:
Epoch:[1/1](1100/112701) loss:1.933 lr:0.0002000 epoch_Time:208.0min:
Epoch:[1/1](1200/112701) loss:1.923 lr:0.0002000 epoch_Time:207.0min:
Epoch:[1/1](1300/112701) loss:1.819 lr:0.0001999 ep

### 模型推理生成文本

In [36]:
import os
import pickle
from contextlib import nullcontext
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import argparse

class TextGenerator:
    def __init__(self, 
                 checkpoint='/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/output/pretrain_768_12_6144.pth',  # 模型检查点路径
                 tokenizer_model_path='/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/tokenizer',  # 分词器模型路径
                 seed=42,  # 随机种子，确保可重复性
                 device=None,  # 设备，优先使用 CUDA，如果没有可用的 CUDA，则使用 CPU
                 dtype="bfloat16"):  # 数据类型，默认为 float32，可以选择 float16 或 bfloat16
        """
        初始化 TextGenerator 类，加载模型、设置设备和分词器等。
        """
        # 模型加载配置
        self.checkpoint = checkpoint  # 保存的模型检查点路径
        self.tokenizer_model_path = tokenizer_model_path  # 分词器模型文件路径
        self.seed = seed  # 随机数种子，用于生成的可重复性
        self.device = device or ('cuda:0' if torch.cuda.is_available() else 'cpu')  # 根据硬件条件选择设备
        self.dtype = dtype  # 模型的浮点数类型
        self.device_type = 'cuda' if 'cuda' in self.device else 'cpu'  # 判断当前设备是否为 CUDA
        
        # 设置随机种子，确保生成的可重复性
        torch.manual_seed(seed)  # 设置 CPU 随机种子
        torch.cuda.manual_seed(seed)  # 设置 CUDA 随机种子
        torch.backends.cuda.matmul.allow_tf32 = True  # 允许 CUDA 使用 TF32 精度进行矩阵乘法运算
        torch.backends.cudnn.allow_tf32 = True  # 允许 cuDNN 使用 TF32 精度加速
        
        # 根据 dtype 选择适当的自动混合精度上下文
        ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[self.dtype]
        self.ctx = nullcontext() if self.device_type == 'cpu' else torch.amp.autocast(device_type=self.device_type, dtype=ptdtype)
        
        # 加载模型检查点文件
        checkpoint_dict = torch.load(self.checkpoint, map_location=self.device)  # 加载模型参数 # 初始化模型参数
        self.model = Llama2(ModelConfig(dim=768, n_layers=12))  # 实例化模型
        sunwanted_prefix = '_orig_mod.'
        for k, v in list(checkpoint_dict.items()):
            if k.startswith(sunwanted_prefix):
                checkpoint_dict[k[len(sunwanted_prefix):]] = checkpoint_dict.pop(k)
        self.model.load_state_dict(checkpoint_dict, strict=False)
        
        # 计算模型参数量
        num_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Model has {num_params / 1e6:.3f} M parameters.")
        # 设置模型为评估模式（evaluation mode），防止训练模式下的 dropout 等操作影响结果
        self.model.eval()
        # 将模型放置到正确的设备上（GPU 或 CPU）
        self.model.to(self.device)
        # 初始化分词器
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_model_path)  # 根据指定的路径加载分词器

    def chat_template(self, prompt):
        message = [
            {"role": "system", "content": "你是一个AI助手，你的名字叫小明。"},
            {"role": "user", "content": prompt}
        ]
        return self.tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

    def sft_sample(self, 
               start="Hello!",  # 生成文本的起始提示词，可以是任意字符串
               num_samples=3,  # 生成样本的数量，默认生成 3 个样本
               max_new_tokens=256,  # 每个样本生成的最大 token 数，默认最多生成 256 个 token
               temperature=0.7,  # 控制生成的随机性，1.0 为标准，值越大越随机
               top_k=300):  # 保留概率最高的 top_k 个 token，限制生成时的选择范围
        """
        根据给定的起始文本生成样本。
        
        :param start: 生成文本的起始提示词
        :param num_samples: 要生成的文本样本数
        :param max_new_tokens: 每个样本生成的最大 token 数
        :param temperature: 控制生成的随机性，值越小生成越确定，值越大生成越随机
        :param top_k: 限制生成时选择的 token 范围
        :return: 生成的文本样本列表
        """
        start = self.chat_template(start)
        # 将起始文本编码为 token id 序列
        start_ids = self.tokenizer(start).data['input_ids']
        # print('start_ids:', start_ids)
        x = (torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...])  # 将编码后的 token id 转为 PyTorch 张量
        generated_texts = []  # 用于保存生成的文本样本
        with torch.no_grad():  # 禁用梯度计算，提升效率
            with self.ctx:  # 进入自动混合精度的上下文（如果是 GPU 并使用 float16 时）
                for k in range(num_samples):  # 循环生成指定数量的样本
                    y = self.model.generate(x, self.tokenizer.eos_token_id, max_new_tokens, temperature=temperature, top_k=top_k)  # 生成文本
                    generated_texts.append(self.tokenizer.decode(y[0].tolist()))  # 解码生成的 token 序列为可读文本
        return generated_texts  # 返回生成的文本样本


    def pretrain_sample(self, 
               start="Hello!",  # 生成文本的起始提示词，可以是任意字符串
               num_samples=3,  # 生成样本的数量，默认生成 3 个样本
               max_new_tokens=256,  # 每个样本生成的最大 token 数，默认最多生成 256 个 token
               temperature=0.7,  # 控制生成的随机性，1.0 为标准，值越大越随机
               top_k=300):  # 保留概率最高的 top_k 个 token，限制生成时的选择范围
        """
        根据给定的起始文本生成样本。
        
        :param start: 生成文本的起始提示词
        :param num_samples: 要生成的文本样本数
        :param max_new_tokens: 每个样本生成的最大 token 数
        :param temperature: 控制生成的随机性，值越小生成越确定，值越大生成越随机
        :param top_k: 限制生成时选择的 token 范围
        :return: 生成的文本样本列表
        """
        # 如果 start 是以 'FILE:' 开头，表示从文件中读取起始文本
        if start.startswith('FILE:'):
            with open(start[5:], 'r', encoding='utf-8') as f:
                start = f.read()  # 读取文件内容作为起始文本
        
        # 将起始文本编码为 token id 序列
        start_ids = self.tokenizer(start).data['input_ids']
        # print('start_ids:', start_ids)
        x = (torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...])  # 将编码后的 token id 转为 PyTorch 张量
        # print(x.shape)
        generated_texts = []  # 用于保存生成的文本样本
        with torch.no_grad():  # 禁用梯度计算，提升效率
            with self.ctx:  # 进入自动混合精度的上下文（如果是 GPU 并使用 float16 时）
                for k in range(num_samples):  # 循环生成指定数量的样本
                    y = self.model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)  # 生成文本
                    generated_texts.append(self.tokenizer.decode(y[0].tolist()))  # 解码生成的 token 序列为可读文本
        
        return generated_texts  # 返回生成的文本样本

In [37]:
print("------------------- Pretrain Sample ------------------- \n")

pretrain_prompt_datas = [
    '<|im_start|>北京大学是',
    '<|im_start|>中国矿业大学（北京）地球科学与测绘工程学院',
]

generator = TextGenerator(checkpoint='/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/output/pretrain_768_12_6144.pth')  # 初始化生成器
for i in range(len(pretrain_prompt_datas)):
    samples = generator.pretrain_sample(start=pretrain_prompt_datas[i], num_samples=1, max_new_tokens=120, temperature=0.75)
    print(f"\nSample {i+1}:\n{pretrain_prompt_datas[i]}{samples[0]}\n{'-'*20}")  # 打印生成的样本并用分隔线分割

print("\n ------------------- SFT Sample ------------------- \n")

sft_prompt_datas = [
    '你好呀',
    "中国的首都是哪里？",
    "1+12等于多少？",
    "你是谁？"
]
generator = TextGenerator(checkpoint='/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo/LLamA2_demo/output/sft_dim768_layers12_vocab_size6144.pth')  # 初始化生成器
for i in range(len(sft_prompt_datas)):
    samples = generator.sft_sample(start=sft_prompt_datas[i], num_samples=1, max_new_tokens=128, temperature=0.6)
    print(f"\nSample {i+1}:\nQuestion: {sft_prompt_datas[i]} \nAI answer: {samples[0]}\n{'-'*20}")  # 打印生成的样本并用分隔线分割

------------------- Pretrain Sample ------------------- 

Model has 82.641 M parameters.

Sample 1:
<|im_start|>北京大学是我国最早成立的国防大学,目光聚焦高科技领域。在10年后,中国人人都想上大学,成为科技的一员,而对于两院院士,他们却力推“国防工程”。
日前,在接受“国家新闻网”采访时,中国工程院院士、中国工程院院士、北京大学教授朱光耀对记者说,中国四大国防中心中,国防工程是最先进的,能打赢各种重大工程,也最有希望迈入世界一流的行列。中国四
--------------------

Sample 2:
<|im_start|>中国矿业大学（北京）地球科学与测绘工程学院图书馆采用平面图、立体图、多媒体图、数字图、便携式图、多媒体图、多媒体应用软件和多媒体信息处理等现代化的图书馆架构,充分利用了图书馆的软硬件资源。图书馆座落在中国矿业大学(北京)太原校区,毗邻中国矿业大学(北京)校区,北依四方山脉,南侧有观音山,北望大运河,是全国唯一一个以采矿区为主的综合性国土资源
--------------------

 ------------------- SFT Sample ------------------- 

Model has 82.641 M parameters.

Sample 1:
Question: 你好呀 
AI answer: 你好,我是小明。有什么我可以帮忙的吗?
--------------------

Sample 2:
Question: 中国的首都是哪里？ 
AI answer: 中国的首都是北京。
--------------------

Sample 3:
Question: 1+12等于多少？ 
AI answer: 2+12等于2。
--------------------

Sample 4:
Question: 你是谁？ 
AI answer: 我不是一个真实的名字,因为我是一台计算机程序,没有身体或者外部特征。我只是一个程序,没有身体或者外部特征。
--------------------
